# Laya multilingual: чистый старт и контроль переобучения

Исходные закреплённые веса Laya; те же E5 top-20 и 22 656 обучающих строк, что у Nano-Jev. Первые 2 272 строки — clean, затем остаток clean/STT в смеси; максимум один проход.

Каждые 142 обновления измеряется только синтетическая валидация. Сохраняются лучшие веса по NLL достижимого правильного ответа: сначала среднее внутри group_id, затем внутри профиля, затем равный вес трём профилям. Остановка после трёх проверок без улучшения минимум 0.005. Baseline тоже участвует в выборе. Точность basic/difficult/ASR публикуется отдельно, включая промахи E5. Личный каталог и test остаются локально.

## Среда

Родной SDK Laya, без генерации текста и замены головы. AMP FP16 для обучения; параметры и AdamW FP32. Валидация FP32.

In [ ]:
import os, sys, json, subprocess
from pathlib import Path
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'laya==0.3.26', 'transformers==5.6.2', 'safetensors==0.6.2', 'accelerate==1.15.0'])
import torch
assert torch.cuda.is_available(), 'Нужен GPU'
print({'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0)})


## Обучение

Encoder lr=1e-5, native choice head lr=3e-5; effective batch 16 через накопление градиентов. Gradient checkpointing уменьшает память. Act/escalate head не обучается. Сохраняется нативное ограничение 48 токенов описания варианта; запрос не обрезается, потери метаданных записываются.

In [ ]:
matches = list(Path('/kaggle/input').rglob('laya-input-manifest.json'))
assert len(matches) == 1
input_dir = matches[0].parent
output_dir = Path('/kaggle/working/results')
command = [sys.executable, str(input_dir / 'train_laya_kaggle.py'), '--input', str(input_dir), '--output', str(output_dir)]
with Path('/kaggle/working/training.log').open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    assert process.wait() == 0, 'См. training.log'


## Выбранные веса

checkpoint — лучшие по синтетической валидации; last-checkpoint — последние. После скачивания нужна локальная оценка на реальном каталоге. Эти вероятности не откалиброваны; оценка точности не подтверждает CPU latency.

In [ ]:
result = json.loads((output_dir / 'result.json').read_text())
print({key: result[key] for key in ('steps', 'best_step', 'stop_reason', 'elapsed_seconds')})
for profile in result['baseline']['metrics']:
    before = result['baseline']['metrics'][profile]['accuracy']
    after = result['best']['metrics'][profile]['accuracy']
    print(f'{profile}: {before:.1%} -> {after:.1%}')
